# Lire le dataset issu de l'étape feature engineering 

1. Chargement

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.decomposition import PCA

df = pd.read_csv('../data/processed/step3_feature_engineering.csv')

print("Shape initial :", df.shape)
df.head()


ModuleNotFoundError: No module named 'imblearn'

2. Vérification

In [ ]:
print("Valeurs manquantes :", df.isnull().sum().sum())

3. Séparer X / y

In [ ]:
X = df.drop('Churn', axis=1)
y = df['Churn']

4. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

5. Encoding

In [ ]:
cat_cols = X_train.select_dtypes(include=['object']).columns

X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)

# aligner colonnes
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

6. Gestion des valeurs infinies / NaN

In [ ]:
X_train.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)

X_train.fillna(X_train.median(numeric_only=True), inplace=True)
X_test.fillna(X_train.median(numeric_only=True), inplace=True)

7. StandardScaler

In [ ]:
scaler = StandardScaler()

num_cols = X_train.select_dtypes(include=['int64','float64']).columns

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

8. Analyse déséquilibre

In [ ]:
print(y.value_counts())

9. SMOTE

In [ ]:

smote = SMOTE(random_state=42)

X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(y_train_resampled.value_counts())

10. PCA

In [ ]:
pca = PCA(n_components=0.95)

X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

print("Shape PCA :", X_train_pca.shape)

11. Sauvegarde

In [ ]:
X_train.to_csv('../data/train_test/X_train.csv', index=False)
X_test.to_csv('../data/train_test/X_test.csv', index=False)
y_train.to_csv('../data/train_test/y_train.csv', index=False)
y_test.to_csv('../data/train_test/y_test.csv', index=False)

print("Données sauvegardées !")
